# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HasanKhan05/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One modeling row = one pseudonymized content item at one monthly decision point.** Daily warehouse facts are aggregated to a page-month before any data enters pandas. Features come only from month `t` and earlier; the observed label comes from month `t+1`. Training anchors are September 2025 through March 2026, validation uses April to predict May, and the sealed test uses May to predict June. A page-month is eligible when the page existed by the decision date, its client has observable GSC coverage in both months, and the feature month has at least 100 impressions. The output is a top-50 monthly human-review queue.

In [1]:
from pathlib import Path
import os

import duckdb
import pandas as pd
from huggingface_hub import get_token

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
extension_dir = repo_root / "work" / "outputs" / ".duckdb_extensions"
extension_dir.mkdir(parents=True, exist_ok=True)

hf_token = os.environ.get("HF_TOKEN") or get_token()
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
assert hf_token, "Add a Hugging Face read token as HF_TOKEN or a Colab secret; never paste it into this notebook."

con = duckdb.connect()
con.execute(f"SET extension_directory='{extension_dir.as_posix()}'")
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [hf_token])

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_march": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

release_summary = pd.DataFrame([
    {"table": "dim_clients", "rows": con.sql(f"SELECT COUNT(*) FROM {TABLES['dim_clients']}").fetchone()[0]},
    {"table": "dim_content", "rows": con.sql(f"SELECT COUNT(*) FROM {TABLES['dim_content']}").fetchone()[0]},
    {"table": "fact_content_daily_performance", "rows": con.sql(f"SELECT COUNT(*) FROM {TABLES['fact_daily']}").fetchone()[0]},
])
date_bounds = con.sql(f"SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date FROM {TABLES['fact_daily']}").df()
for row in release_summary.itertuples(index=False):
    print(f"{row.table}: {row.rows:,} rows")
print(f"Daily date window: {date_bounds.loc[0, 'min_date']} to {date_bounds.loc[0, 'max_date']}")
print("Decision grain: one pseudonymized content item at one monthly anchor")
print("Windows: train 2025-09..2026-03 | validation 2026-04 -> 2026-05 | sealed 2026-05 -> 2026-06")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

dim_clients: 104 rows
dim_content: 519,606 rows
fact_content_daily_performance: 78,835,655 rows
Daily date window: 2025-01-27 00:00:00 to 2026-06-30 00:00:00
Decision grain: one pseudonymized content item at one monthly anchor
Windows: train 2025-09..2026-03 | validation 2026-04 -> 2026-05 | sealed 2026-05 -> 2026-06


## 2. Fields: feature / label / context / excluded

Every field belongs to exactly one bucket. Features must be knowable by the monthly decision point. The label is a later observed outcome, context fields support joins/splits/reading, and excluded fields are future, private, circular, or too risky for this first model. IDs remain context only.

In [2]:
field_contract = pd.DataFrame([
    ("feature", "impressions, clicks, CTR, valid weighted position, active days", "aggregated from feature month t"),
    ("feature", "prior-month impressions/clicks and pre-decision momentum", "known before the decision point"),
    ("feature", "content age, content type, search volume, missingness flags", "safe structured metadata available by t"),
    ("label", "future_decline", "1 when month t+1 impressions are below 80% of month t"),
    ("label", "outcome_impressions", "source measurement for future_decline; never a feature"),
    ("context", "client_hash_id, content_hash_id, month", "joins, grouped checks, time splits, and queue reference only"),
    ("context", "GSC/GA4 availability flags", "measurement coverage and honest filtering"),
    ("excluded", "all target-month metrics", "future information would leak the answer"),
    ("excluded", "IDs as predictive features", "pseudonym codes have no portable meaning"),
    ("excluded", "raw names, domains, URLs, queries, titles", "private and absent from the public release"),
    ("excluded", "provider_used and model_used", "not required for the editorial decision"),
    ("excluded", "fact_content_query_90d fields", "fixed window overlaps final months and raises leakage risk"),
], columns=["bucket", "fields", "rule_or_reason"])
field_contract


,bucket,fields,rule_or_reason
0,feature,"impressions, clicks, CTR, valid weighted posit...",aggregated from feature month t
1,feature,prior-month impressions/clicks and pre-decisio...,known before the decision point
2,feature,"content age, content type, search volume, miss...",safe structured metadata available by t
3,label,future_decline,1 when month t+1 impressions are below 80% of ...
4,label,outcome_impressions,source measurement for future_decline; never a...
5,context,"client_hash_id, content_hash_id, month","joins, grouped checks, time splits, and queue ..."
6,context,GSC/GA4 availability flags,measurement coverage and honest filtering
7,excluded,all target-month metrics,future information would leak the answer
8,excluded,IDs as predictive features,pseudonym codes have no portable meaning
9,excluded,"raw names, domains, URLs, queries, titles",private and absent from the public release


## 3. Verify it with queries (grain, counts, missing values, windows)

The following probes use the March 2026 mid-panel partition for query development, not the final-month sample. The full-table count and bounds above reconcile with the published release. The grain probe must return zero duplicates. Availability is checked as three-valued data because `TRUE`, `FALSE`, and `NULL` have different meanings.

In [3]:
march_checks = con.sql(f"""
    SELECT COUNT(*) AS rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS clients,
           COUNT(DISTINCT content_hash_id) AS content_items,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_true_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS gsc_false_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS NULL) AS gsc_null_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_true_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_not_true_rows
    FROM {TABLES['fact_march']}
""").df()
duplicate_grain = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS duplicate_count
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
content_missingness = con.sql(f"""
    SELECT COUNT(*) AS rows,
           COUNT(*) FILTER (WHERE content_created_date IS NULL) AS missing_created_date,
           COUNT(*) FILTER (WHERE content_type IS NULL) AS missing_content_type,
           COUNT(*) FILTER (WHERE search_volume IS NULL) AS missing_search_volume
    FROM {TABLES['dim_content']}
""").df()
print("March 2026 mid-panel checks:")
display(march_checks)
print(f"Duplicate rows at report_date × client × content grain: {len(duplicate_grain)}")
assert duplicate_grain.empty, "Daily fact grain is not unique"
print("Content metadata missingness:")
display(content_missingness)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 mid-panel checks:


,rows,min_date,max_date,clients,content_items,gsc_true_rows,gsc_false_rows,gsc_null_rows,ga4_true_rows,ga4_not_true_rows
0,9841378,2026-03-01,2026-03-31,55,331437,3611061,6230317,0,413966,9427412


Duplicate rows at report_date × client × content grain: 0
Content metadata missingness:


,rows,missing_created_date,missing_content_type,missing_search_volume
0,519606,0,0,142622


## 4. Data limits

This is an unbalanced observational panel: clients start tracking on different dates, early rows may contain GSC without GA4, and missing availability is not the same as zero activity. The data can show measured future movement and ranking performance, but it cannot prove that a refresh caused recovery or reveal Google's ranking algorithm. Pages absent in an otherwise active outcome month may represent zero visibility, removal, or measurement gaps, so that case is counted and disclosed. The fixed query-table window overlaps the final months and is excluded. June 2026 remains sealed until model selection is frozen.

In [4]:
coverage = con.sql(f"""
    SELECT COUNT(*) AS clients,
           COUNT(*) FILTER (WHERE gsc_data_start IS NULL) AS missing_gsc_start,
           COUNT(*) FILTER (WHERE ga4_data_start IS NULL) AS missing_ga4_start,
           MIN(gsc_data_start) AS earliest_gsc_start,
           MAX(gsc_data_start) AS latest_gsc_start,
           COUNT(*) FILTER (WHERE gsc_data_start <= DATE '2025-09-01') AS eligible_for_full_training_window
    FROM {TABLES['dim_clients']}
""").df()
display(coverage)
print("Interpretation: coverage differs by client; zero-filled or NULL analytics rows are never treated as measured zero engagement.")
print("Claim boundary: observed, measured, directional, and decision-support only.")


,clients,missing_gsc_start,missing_ga4_start,earliest_gsc_start,latest_gsc_start,eligible_for_full_training_window
0,104,37,53,2025-01-27,2026-06-02,16


Interpretation: coverage differs by client; zero-filled or NULL analytics rows are never treated as measured zero engagement.
Claim boundary: observed, measured, directional, and decision-support only.


## 5. Self-check

- [x] The unit of analysis and all feature/label windows are explicit.
- [x] Every planned field belongs to feature, label, context, or excluded.
- [x] Counts, date bounds, grain, missingness, and coverage claims have executed queries.
- [x] The March mid-panel partition—not the final-month sample—was used for query development.
- [x] IDs are context only; future fields and fixed-window query totals are excluded.
- [x] No client names, domains, URLs, private queries, credentials, or identifying examples appear.
- [x] Claims use observed, measured, directional, and decision-support language.
- [x] The notebook has been run top to bottom with no errors before submission.